# CS156: Pipeline - Second Draft

**Can You Diagnose Autism From Brain Noise?**

There’s a persistent idea in computational neuroscience that you can look at a brain scan and read *something* meaningful: a diagnosis, a trait, or a difference in how someone thinks. It’s a compelling idea. It’s also, at best, an approximation.

This project starts from a deliberately uncomfortable question:

> If you take a noisy, indirect measurement of brain activity and reduce it to correlations between ~200 regions, is there actually enough signal left to distinguish individuals with Autism Spectrum Disorder (ASD) from controls?

The data used here are resting-state fMRI time series, summarized as functional connectivity matrices. In practice, this means:
- we are not observing neural activity directly  
- we are not measuring interactions between regions  
- we are measuring statistical co-variation under a long chain of preprocessing assumptions  

So the object we feed into our models is already several steps removed from anything we might confidently call “brain function.”

Given that, the goal of this project is not to build a diagnostic tool, but rather to test how different modeling assumptions behave when applied to this representation.

Each subject is represented as a connectivity matrix derived from ROI time series (`rois_cc200`). From there, we ask a simple but pointed question:

> Does treating the brain as a graph actually help, or are we just adding structure to noise?

To answer this, we compare four approaches:

* Logistic Regression on flattened connectivity features  
* A multilayer perceptron (MLP) on the same representation  
* A graph convolutional network (GCN) that assumes connectivity structure matters  
* A graph attention network (GAT) that learns which connections to emphasize  

All models operate on the same underlying data, allowing differences in performance to reflect model assumptions rather than differences in input.

If graph-based models outperform simpler baselines, that suggests there is meaningful relational structure in the data. If they do not, it raises a more uncomfortable possibility: that increasing model sophistication does not recover signal that may not be there to begin with.

The notebook proceeds as follows:

1. Load and verify the curated dataset  
2. Construct functional connectivity matrices  
3. Prepare inputs for each modeling approach  
4. Train and evaluate models under consistent conditions  
5. Compare results and analyze differences  

The aim is not to settle whether functional connectivity can diagnose ASD, but to probe how far this representation can be pushed—and where it breaks down.

# Data

## Data Source

## Preprocessing Done on Original Data

# Loading the Data and Feature Engineering

In [ ]:
# loading data and building correlation matrices
import numpy as np
import pandas as pd
from pathlib import Path

base_dir = Path("data") / "abide_fmri"
ts_dir = base_dir / "timeseries"
fc_dir = base_dir / "connectivity_matrices"
fc_dir.mkdir(parents=True, exist_ok=True)

subjects_path = base_dir / "subjects_clean.csv"
if subjects_path.exists():
    subjects_df = pd.read_csv(subjects_path)
    file_ids = subjects_df["FILE_ID"].dropna().astype(str).drop_duplicates().tolist()
else:
    # Fallback: use every .1D file present if label file is unavailable
    file_ids = sorted(p.stem for p in ts_dir.glob("*.1D"))

print(f"Candidate subjects: {len(file_ids)}")

valid_ids = []
failed_ids = []
roi_count = None

for fid in file_ids:
    ts_path = ts_dir / f"{fid}.1D"
    if not ts_path.exists():
        failed_ids.append(fid)
        continue

    try:
        ts = np.loadtxt(ts_path)
        if ts.ndim != 2:
            failed_ids.append(fid)
            continue

        # Expect shape: (timepoints, regions). If flipped, transpose.
        if ts.shape[1] == 200:
            pass  # correct
        elif ts.shape[0] == 200:
            ts = ts.T
        else:
            failed_ids.append(fid)
            continue

        corr = np.corrcoef(ts, rowvar=False)
        corr = np.nan_to_num(corr).astype(np.float32)
        corr = np.abs(corr)
        np.fill_diagonal(corr, 0)

        if roi_count is None:
            roi_count = corr.shape[0]
        if corr.shape != (roi_count, roi_count):
            failed_ids.append(fid)
            continue

        np.save(fc_dir / f"{fid}.npy", corr)
        valid_ids.append(fid)
    except Exception:
        failed_ids.append(fid)

index_df = pd.DataFrame({"FILE_ID": valid_ids})
if subjects_path.exists():
    subjects_clean = subjects_df[subjects_df["FILE_ID"].isin(valid_ids)].copy()
    subjects_clean.to_csv(base_dir / "subjects_with_fc.csv", index=False)
    index_df = index_df.merge(subjects_clean[["FILE_ID", "DX_GROUP"]], on="FILE_ID", how="left")

index_df.to_csv(base_dir / "connectivity_index.csv", index=False)

print(f"Saved FC matrices: {len(valid_ids)}")
print(f"Failed subjects: {len(failed_ids)}")
if roi_count is not None:
    print(f"Matrix shape per subject: ({roi_count}, {roi_count})")
if failed_ids:
    print("Example failures:", failed_ids[:10])

In [ ]:
# Build sparse PyG graphs from connectivity matrices (top-k edges per node)
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.data import Data

base_dir = Path("data") / "abide_fmri"
fc_dir = base_dir / "connectivity_matrices"
index_path = base_dir / "connectivity_index.csv"
pyg_dir = base_dir / "pyg"
pyg_dir.mkdir(parents=True, exist_ok=True)

TOP_K = 10

def matrix_to_topk_edge_index(adj: np.ndarray, k: int = 10):
    """Return undirected sparse edges from a dense adjacency matrix."""
    n = adj.shape[0]
    a = np.array(adj, dtype=np.float32, copy=True)
    np.fill_diagonal(a, 0.0)

    undirected_edges = {}  # (u, v) -> weight
    for i in range(n):
        row = a[i]
        nonzero = np.flatnonzero(row > 0)
        if nonzero.size == 0:
            continue

        keep = min(k, nonzero.size)
        top_idx_local = np.argpartition(row[nonzero], -keep)[-keep:]
        nbrs = nonzero[top_idx_local]

        for j in nbrs:
            u, v = (i, j) if i < j else (j, i)
            if u == v:
                continue
            w = float(row[j])
            if w > undirected_edges.get((u, v), 0.0):
                undirected_edges[(u, v)] = w

    if not undirected_edges:
        return torch.empty((2, 0), dtype=torch.long), torch.empty((0,), dtype=torch.float32)

    edges = []
    weights = []
    for (u, v), w in undirected_edges.items():
        edges.append([u, v])
        edges.append([v, u])
        weights.append(w)
        weights.append(w)

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    return edge_index, edge_weight

index_df = pd.read_csv(index_path)
label_map = {1: 1, 2: 0}  # ASD=1, Control=0
graphs = []
skipped = []

for _, row in index_df.iterrows():
    fid = str(row["FILE_ID"])
    y_raw = row.get("DX_GROUP", np.nan)
    mat_path = fc_dir / f"{fid}.npy"
    if not mat_path.exists():
        skipped.append(fid)
        continue

    adj = np.load(mat_path)
    if adj.ndim != 2 or adj.shape[0] != adj.shape[1]:
        skipped.append(fid)
        continue

    edge_index, edge_weight = matrix_to_topk_edge_index(adj, k=TOP_K)
    num_nodes = adj.shape[0]

    # Simple node features: identity matrix (one-hot node identity)
    x = torch.eye(num_nodes, dtype=torch.float32)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        edge_weight=edge_weight,
        file_id=fid
    )

    if pd.notna(y_raw):
        data.y = torch.tensor([label_map.get(int(y_raw), -1)], dtype=torch.long)

    graphs.append(data)

torch.save(graphs, pyg_dir / f"graphs_top{TOP_K}.pt")

print(f"Saved {len(graphs)} PyG graphs to: {pyg_dir / f'graphs_top{TOP_K}.pt'}")
if graphs:
    avg_edges = int(np.mean([g.edge_index.shape[1] for g in graphs]))
    print(f"Average directed edges per graph: {avg_edges}")
    print(f"Example graph: nodes={graphs[0].num_nodes}, edges={graphs[0].edge_index.shape[1]}")
if skipped:
    print(f"Skipped {len(skipped)} subjects (missing/invalid matrix).")

# Exploratory Data Analysis

# Splitting the Data

# Model Selection

## Baseline: Predicting the Majority Class

## Logistic Regression 

## Multilayer Perceptron (MLP) 

## Graph Convolutional Network (GCN)  

### Mathematical Basis of GCNs

## Graph Attention Network (GAT)

### Mathematical Basis of GATs

# Training with Cross-Validation and Hyperparameter Tuning

# Evaluation on the Test Set

# Results and Conclusions 

# Limitations 

# Executive Summary

# AI Statement

# References